# Agente de Memoria (Juego de Parejas)



In [1]:
import random
import math


In [2]:
class JuegoMemoria:
    def __init__(self, n_parejas):
        self.n_parejas = n_parejas
        self.baraja = [i for i in range(n_parejas) for _ in range(2)]
        random.shuffle(self.baraja)
        self.emparejadas = [False] * (2 * n_parejas)

    def revelar(self, pos):
        return self.baraja[pos]

    def emparejar(self, a, b):
        if self.baraja[a] == self.baraja[b]:
            self.emparejadas[a] = True
            self.emparejadas[b] = True
            return True
        return False

    def todo_emparejado(self):
        return all(self.emparejadas)


## El agente recuerda, explora y explota
- Recuerda: guarda en MEMORIA una lista de posiciones para cada valor descubierto.
- Explora: selecciona posiciones no vistas usando UCB (Upper Confidence Bound) para preferir posiciones poco visitadas.
- Explota: si conoce dos posiciones con el mismo valor las juega inmediatamente para completar la pareja.

Flujo simple: primero busca parejas conocidas; si no hay, explora una posición desconocida con UCB, revela otra posición buscando su pareja en la memoria; actualiza estadísticas por posición.


In [3]:
class AgenteMemoria:
    def __init__(self, n_posiciones, c=0.5):
        self.n = n_posiciones
        self.c = c
        self.memoria = {}
        self.conteos = [0] * n_posiciones
        self.est = [0.0] * n_posiciones
        self.total = 0
        self.no_vistas = set(range(n_posiciones))
        self.pos_emparejadas = set()

    def observar(self, pos, valor):
        if pos in self.pos_emparejadas:
            return
        if valor not in self.memoria:
            self.memoria[valor] = []
        if pos not in self.memoria[valor]:
            self.memoria[valor].append(pos)
        if pos in self.no_vistas:
            self.no_vistas.remove(pos)

    def olvidar_emparejadas(self, valor):
        if valor in self.memoria:
            self.memoria[valor] = [p for p in self.memoria[valor] if p not in self.pos_emparejadas]
            if not self.memoria[valor]:
                del self.memoria[valor]

    def actualizar_estadisticas(self, posiciones, recompensa):
        for p in posiciones:
            self.conteos[p] += 1
            self.est[p] += (recompensa - self.est[p]) / self.conteos[p]
        self.total += 1

    def seleccionar_ucb(self, candidatos):
        candidatos = list(candidatos)
        for p in candidatos:
            if self.conteos[p] == 0:
                return p
        mejor = None
        mejor_val = -1e9
        logt = math.log(max(1, self.total))
        for p in candidatos:
            val = self.est[p] + self.c * math.sqrt(logt / self.conteos[p])
            if val > mejor_val:
                mejor_val = val
                mejor = p
        return mejor

    def jugar_turno(self, juego):
        for v, poses in list(self.memoria.items()):
            poses = [p for p in poses if p not in self.pos_emparejadas]
            if len(poses) >= 2:
                a, b = poses[0], poses[1]
                va = juego.revelar(a)
                vb = juego.revelar(b)
                matched = juego.emparejar(a, b)
                if matched:
                    self.pos_emparejadas.update([a, b])
                    self.olvidar_emparejadas(v)
                    self.actualizar_estadisticas([a, b], 1)
                else:
                    self.observar(a, va)
                    self.observar(b, vb)
                    self.actualizar_estadisticas([a, b], 0)
                return

        if self.no_vistas:
            a = self.seleccionar_ucb(self.no_vistas)
        else:
            candidatos = [i for i in range(self.n) if i not in self.pos_emparejadas]
            a = self.seleccionar_ucb(candidatos)

        va = juego.revelar(a)
        self.observar(a, va)

        conocidos = [p for p in self.memoria.get(va, []) if p != a and p not in self.pos_emparejadas]
        if conocidos:
            b = conocidos[0]
        else:
            restantes = set(range(self.n)) - {a} - self.pos_emparejadas
            if restantes:
                no_vistas_restantes = restantes & self.no_vistas
                if no_vistas_restantes:
                    b = self.seleccionar_ucb(no_vistas_restantes)
                else:
                    b = self.seleccionar_ucb(restantes)
            else:
                return

        vb = juego.revelar(b)
        self.observar(b, vb)
        matched = juego.emparejar(a, b)
        if matched:
            self.pos_emparejadas.update([a, b])
            self.olvidar_emparejadas(va)
            self.olvidar_emparejadas(vb)
            self.actualizar_estadisticas([a, b], 1)
        else:
            self.actualizar_estadisticas([a, b], 0)


In [4]:
def simular(n_parejas, episodios=100, seed=None):
    if seed is not None:
        random.seed(seed)
    totales = []
    for _ in range(episodios):
        juego = JuegoMemoria(n_parejas)
        agente = AgenteMemoria(2 * n_parejas)
        turnos = 0
        while not juego.todo_emparejado():
            agente.jugar_turno(juego)
            turnos += 1
            if turnos > 1000:
                break
        totales.append(turnos)
    return sum(totales) / len(totales)

avg = simular(8, episodios=200, seed=42)
print('turnos promedio', avg)

turnos promedio 12.345


## Explicacion

Política la regla que el agente usa para elegir qué cartas voltear. Priorizá parejas que ya recuerda; si no hay, busca cartas nuevas de forma inteligente.

Función de valor una estimación simple que dice cuánto conviene elegir una posición; se actualiza con lo que pasa cuando prueba esa posición.

Recompensa la señal que indica si una jugada fue buena (por ejemplo +1 por emparejar, 0 por fallar); con ella el agente aprende qué posiciones son mejores.

UCB una fórmula que combina la estimación de valor y la incertidumbre para balancear explorar y explotar; ayuda a elegir posiciones poco probadas pero potencialmente buenas.

En resumen: el agente recuerda lo que vio, usa esa memoria para aprovechar parejas conocidas y aplica UCB para explorar cuando falta información.
